In [2]:
import pandas as pd

players = pd.read_csv("data/players.csv", parse_dates=["date_of_birth", "contract_expiration_date"])
apps = pd.read_csv("data/appearances.csv", parse_dates=["date"])

print(players.shape, apps.shape)
print("Første kamp:", apps["date"].min().date(), " Siste kamp:", apps["date"].max().date())

(50149, 26) (1894350, 13)
Første kamp: 2012-07-03  Siste kamp: 2026-06-28


In [3]:
season = apps[(apps["date"] >= "2025-07-01") & (apps["date"] <= "2026-06-30")]
print("Kamper i sesongen:", len(season))
print("Unike spillere:", season["player_id"].nunique())

Kamper i sesongen: 156676
Unike spillere: 9051


In [4]:
stats = season.groupby("player_id").agg(
    games=("game_id", "count"),
    minutes=("minutes_played", "sum"),
    goals=("goals", "sum"),
    assists=("assists", "sum"),
    yellow=("yellow_cards", "sum"),
)

cl = season[season["competition_id"] == "CL"].groupby("player_id")["minutes_played"].sum()
stats["cl_minutes"] = cl.reindex(stats.index).fillna(0)

print(stats.shape)
print("Spillere med CL-minutter:", (stats["cl_minutes"] > 0).sum())
stats.head()

(9051, 6)
Spillere med CL-minutter: 834


,games,minutes,goals,assists,yellow,cl_minutes
player_id,,,,,,
3333,22,946,1,1,2,0.0
5336,8,288,0,0,1,0.0
7161,28,1288,3,3,3,81.0
8198,3,270,2,0,0,0.0
11530,24,1902,0,0,4,0.0


In [5]:
df = players[players["last_season"] == 2025].merge(stats, left_on="player_id", right_index=True)
print("Etter merge:", df.shape)

Etter merge: (9035, 32)


In [6]:
ref = pd.Timestamp("2026-06-30")
df["age"] = (ref - df["date_of_birth"]).dt.days / 365.25
df["contract_years_left"] = (df["contract_expiration_date"] - ref).dt.days / 365.25

df[["name", "age", "contract_years_left"]].head()

,name,age,contract_years_left
189,James Milner,40.484600,NaN
418,Anastasios Tsokanis,35.162218,0.999316
645,Jonas Hofmann,33.960301,0.999316
760,Cristiano Ronaldo,41.396304,0.999316
915,Lionel Carole,35.216975,NaN


In [7]:
df = df[(df["minutes"] >= 450) & (df["market_value_in_eur"] > 0)].copy()

df["goals_per90"] = df["goals"] / df["minutes"] * 90
df["assists_per90"] = df["assists"] / df["minutes"] * 90

print("Endelig datasett:", df.shape)

Endelig datasett: (5391, 36)


In [8]:
print(df.isna().sum()[df.isna().sum() > 0])
df[["age", "contract_years_left", "minutes", "goals_per90", "assists_per90", "cl_minutes", "market_value_in_eur"]].describe()

first_name                               284
country_of_birth                         144
city_of_birth                            144
foot                                      46
height_in_cm                              39
contract_expiration_date                 551
agent_name                              1377
international_caps                       853
international_goals                      853
current_national_team_id                5268
current_club_domestic_competition_id      81
current_club_name                         81
contract_years_left                      551
dtype: int64


,age,contract_years_left,minutes,goals_per90,assists_per90,cl_minutes,market_value_in_eur
count,5391.000000,4840.000000,5391.000000,5391.000000,5391.000000,5391.000000,5.391000e+03
mean,26.973634,2.231325,1751.284363,0.127073,0.098370,64.555185,7.613754e+06
std,4.306022,1.209865,938.054091,0.165803,0.112177,196.162332,1.437870e+07
min,16.495551,-0.082136,450.000000,0.000000,0.000000,0.000000,2.500000e+04
25%,23.690623,0.999316,928.500000,0.000000,0.000000,0.000000,8.000000e+05
50%,26.477755,2.001369,1625.000000,0.067720,0.069085,0.000000,2.500000e+06
75%,29.797399,3.000684,2432.000000,0.188186,0.156307,0.000000,8.000000e+06
max,42.699521,8.999316,5211.000000,1.272601,0.788486,1440.000000,2.000000e+08


In [9]:
df.to_csv("data/model_data.csv", index=False)
print("Lagret", len(df), "rader")

Lagret 5391 rader
